# VQA Project - Getting Started

This notebook provides a quick start guide for the Visual Question Answering project.

## Setup

First, make sure you have installed all dependencies:
```bash
pip install -r ../requirements.txt
```

In [ ]:
import sys
sys.path.append('../src')

import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt

from dataset import Vocabulary, VQADataset
from model import EncoderCNN, DecoderRNN, VQAModel
import utils

## 1. Creating a Vocabulary

The vocabulary handles tokenization and encoding/decoding of text.

In [ ]:
# Create vocabulary
vocab = Vocabulary()

# Example sentences
sentences = [
    "What color is the car?",
    "How many people are in the image?",
    "Is there a dog in the picture?",
    "What is the person doing?"
]

# Build vocabulary from sentences
vocab.build_vocabulary(sentences, min_word_freq=1)

print(f"Vocabulary size: {len(vocab)}")
print(f"\nSpecial tokens:")
print(f"  PAD: {vocab.word2idx[vocab.pad_token]}")
print(f"  UNK: {vocab.word2idx[vocab.unk_token]}")
print(f"  START: {vocab.word2idx[vocab.start_token]}")
print(f"  END: {vocab.word2idx[vocab.end_token]}")

## 2. Encoding and Decoding Text

In [ ]:
# Test encoding
text = "What color is the car?"
encoded = vocab.encode(text, max_length=10)
decoded = vocab.decode(encoded)

print(f"Original: {text}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")

## 3. Model Architecture

Let's create and inspect the VQA model.

In [ ]:
# Create model
model = VQAModel(
    vocab_size=len(vocab),
    embed_dim=256,
    hidden_dim=512,
    image_feature_dim=2048,
    num_layers=2,
    dropout=0.5,
    pretrained_encoder=False,  # Set to True when you have internet
    freeze_encoder=True
)

# Print model summary
utils.print_model_summary(model)

## 4. Testing Forward Pass

Test the model with dummy data to ensure shapes are correct.

In [ ]:
# Create dummy data
batch_size = 4
question_length = 10

# Dummy images (batch_size, 3, 224, 224)
dummy_images = torch.randn(batch_size, 3, 224, 224)

# Dummy questions (batch_size, question_length)
dummy_questions = torch.randint(0, len(vocab), (batch_size, question_length))

# Forward pass
with torch.no_grad():
    outputs = model(dummy_images, dummy_questions)

print(f"Input shapes:")
print(f"  Images: {dummy_images.shape}")
print(f"  Questions: {dummy_questions.shape}")
print(f"\nOutput shape: {outputs.shape}")
print(f"Expected: (batch_size={batch_size}, question_length={question_length}, vocab_size={len(vocab)})")

## 5. Next Steps

Now that you've verified the basic components work:

1. **Download a VQA dataset** (e.g., VQA v2, COCO-QA)
2. **Prepare your data** in the `data/` directory
3. **Update the dataset paths** in `train.py`
4. **Start training**: `python src/train.py --data_path data/`

### Recommended VQA Datasets:
- **VQA v2**: https://visualqa.org/download.html
- **COCO-QA**: https://www.cs.toronto.edu/~mren/research/imageqa/data/cocoqa/
- **Visual7W**: http://web.stanford.edu/~yukez/visual7w/

### Useful Resources:
- PyTorch Documentation: https://pytorch.org/docs/
- VQA Paper: https://arxiv.org/abs/1505.00468
- Attention Mechanisms: https://arxiv.org/abs/1409.0473

## Troubleshooting

If you encounter issues:

1. **NLTK data error**: Run `python -c "import nltk; nltk.download('punkt')"`
2. **CUDA out of memory**: Reduce `batch_size` in training
3. **Import errors**: Make sure you're in the correct directory